# Simple test for manual B/Omega benchmark
快速验证 `cd_A` / `cd_B` / `cd_BOmega` 三个算法是否可在人工场景上正常运行。

In [7]:
import os
import sys
import argparse

cwd = os.getcwd()
repo_root = cwd if os.path.basename(cwd) != 'experiments' else os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.append(repo_root)

from experiments.test_manual_B_Omega_cd_algorithms import (
    get_manual_experiments,
    run_one_experiment,
    save_trial_csv,
    save_summary_csv,
)

In [8]:
args = argparse.Namespace(
    n_repeats=20,
    seed=0,
    data_seed=42,
    threshold=0.05,
    k=None,
    dag_tol=1e-8,
    epochs_a=500,
    epochs_b=500,
    epochs_bomega=500,
    lambda_l0=0.0,
    tol=1e-4,
    patience=10,
    min_epochs=100,
    eps_omega=1e-8,
    outdir=os.path.join('experiments', 'results'),
    tag='manual_B_Omega_cd_simple_test',
)

experiments = get_manual_experiments()
for exp in experiments:
    print(f'Experiment: {exp["name"]}')
    rows = run_one_experiment(exp, n_samples=10000, args=args)
    print(f'rows generated: {len(rows)}\n')

Experiment: d=3, A→B←C
Running: d=3, A→B←C
cd_A       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.337s
cd_B       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.255s
cd_BOmega  Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.286s
rows generated: 60

Experiment: d=3, A→B→C
Running: d=3, A→B→C
cd_A       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.343s
cd_B       Correct rate=0.550 | CPDAG_SHD=0.45 | Time=0.258s
cd_BOmega  Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.301s
rows generated: 60

Experiment: d=3, A→B→C + A→C
Running: d=3, A→B→C + A→C
cd_A       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.290s
cd_B       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.224s
cd_BOmega  Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.259s
rows generated: 60

Experiment: d=4, A→B, B→C, B→D
Running: d=4, A→B, B→C, B→D
cd_A       Correct rate=0.500 | CPDAG_SHD=1.00 | Time=0.597s
cd_B       Correct rate=0.200 | CPDAG_SHD=1.60 | Time=0.464s
cd_BOmega  Correct rate=0.350 | CPDAG_SHD=1.15 | Time=0.487s
rows gen

In [10]:
import pandas as pd

all_rows = []

for exp in experiments:
    exp_rows = run_one_experiment(exp, n_samples=10000, args=args)
    all_rows.extend(exp_rows)

# 明细表
results_df = pd.DataFrame([row.__dict__ for row in all_rows])
results_df = results_df.sort_values(["experiment", "algorithm", "repeat_id"]).reset_index(drop=True)

# 汇总表（每个实验 × 每个算法）
summary_df = (
    results_df
    .groupby(["experiment", "algorithm"], as_index=False)
    .agg(
        repeats=("repeat_id", "count"),
        mec_rate=("mec_match", "mean"),
        exact_rate=("exact_match", "mean"),
        cpdag_shd_mean=("cpdag_shd", "mean"),
        cpdag_shd_std=("cpdag_shd", "std"),
        runtime_mean_sec=("runtime_sec", "mean"),
        runtime_std_sec=("runtime_sec", "std"),
    )
    .sort_values(["experiment", "algorithm"])
    .reset_index(drop=True)
)

summary_df[["mec_rate", "exact_rate", "cpdag_shd_mean", "cpdag_shd_std", "runtime_mean_sec", "runtime_std_sec"]] = (
    summary_df[["mec_rate", "exact_rate", "cpdag_shd_mean", "cpdag_shd_std", "runtime_mean_sec", "runtime_std_sec"]].round(4)
)

print(f"Total rows: {len(results_df)}")
print("\nDetail table (first 20 rows):")
display(results_df.head(20))

print("\nSummary table:")
display(summary_df)


Running: d=3, A→B←C
cd_A       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.270s
cd_B       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.209s
cd_BOmega  Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.242s
Running: d=3, A→B→C
cd_A       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.293s
cd_B       Correct rate=0.550 | CPDAG_SHD=0.45 | Time=0.239s
cd_BOmega  Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.275s
Running: d=3, A→B→C + A→C
cd_A       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.304s
cd_B       Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.252s
cd_BOmega  Correct rate=1.000 | CPDAG_SHD=0.00 | Time=0.281s
Running: d=4, A→B, B→C, B→D
cd_A       Correct rate=0.500 | CPDAG_SHD=1.00 | Time=0.657s
cd_B       Correct rate=0.200 | CPDAG_SHD=1.60 | Time=0.538s
cd_BOmega  Correct rate=0.350 | CPDAG_SHD=1.15 | Time=0.569s
Running: d=4, A→C, A→D, B→C, B→D
cd_A       Correct rate=0.800 | CPDAG_SHD=1.20 | Time=0.498s
cd_B       Correct rate=0.650 | CPDAG_SHD=2.10 | Time=0.352s
cd_BOmega  Correct 

,n_samples,experiment,algorithm,repeat_id,seed,mec_match,exact_match,cpdag_shd,runtime_sec
0,10000,"d=3, A→B←C",cd_A,0,0,1,1,0.0,0.285593
1,10000,"d=3, A→B←C",cd_A,1,1,1,1,0.0,0.261947
2,10000,"d=3, A→B←C",cd_A,2,2,1,1,0.0,0.301665
3,10000,"d=3, A→B←C",cd_A,3,3,1,1,0.0,0.247169
4,10000,"d=3, A→B←C",cd_A,4,4,1,1,0.0,0.253172
5,10000,"d=3, A→B←C",cd_A,5,5,1,1,0.0,0.279413
6,10000,"d=3, A→B←C",cd_A,6,6,1,1,0.0,0.264977
7,10000,"d=3, A→B←C",cd_A,7,7,1,1,0.0,0.266373
8,10000,"d=3, A→B←C",cd_A,8,8,1,1,0.0,0.268348
9,10000,"d=3, A→B←C",cd_A,9,9,1,1,0.0,0.265293



Summary table:


,experiment,algorithm,repeats,mec_rate,exact_rate,cpdag_shd_mean,cpdag_shd_std,runtime_mean_sec,runtime_std_sec
0,"d=3, A→B←C",cd_A,20,1.00,1.00,0.00,0.0000,0.2698,0.0158
1,"d=3, A→B←C",cd_B,20,1.00,1.00,0.00,0.0000,0.2090,0.0093
2,"d=3, A→B←C",cd_BOmega,20,1.00,1.00,0.00,0.0000,0.2424,0.0103
3,"d=3, A→B→C",cd_A,20,1.00,0.75,0.00,0.0000,0.2935,0.0187
4,"d=3, A→B→C",cd_B,20,0.55,0.30,0.45,0.5104,0.2391,0.0326
5,"d=3, A→B→C",cd_BOmega,20,1.00,0.75,0.00,0.0000,0.2753,0.0230
6,"d=3, A→B→C + A→C",cd_A,20,1.00,0.75,0.00,0.0000,0.3043,0.0158
7,"d=3, A→B→C + A→C",cd_B,20,1.00,0.70,0.00,0.0000,0.2525,0.0218
8,"d=3, A→B→C + A→C",cd_BOmega,20,1.00,1.00,0.00,0.0000,0.2813,0.0167
9,"d=4, A→B, B→C, B→D",cd_A,20,0.50,0.30,1.00,1.4142,0.6568,0.0970


In [11]:
# 简洁版矩阵表：横轴=9个实验，纵轴=算法，值=mec_rate
mec_matrix = (
    summary_df
    .pivot(index='algorithm', columns='experiment', values='mec_rate')
)

# 若列名中包含“d=...”，按数字维度顺序稳定排序；否则保持原顺序
mec_matrix = mec_matrix.reindex(sorted(mec_matrix.columns), axis=1)

print('MEC rate matrix (rows=algorithm, cols=experiment):')
display(mec_matrix.round(4))

MEC rate matrix (rows=algorithm, cols=experiment):


experiment,"d=3, A→B←C","d=3, A→B→C","d=3, A→B→C + A→C","d=4, A→B, B→C, B→D","d=4, A→C, A→D, B→C, B→D","d=4, A→D, B→D, C→D","d=5, e=4, |v|=0","d=5, e=4, |v|=1","d=5, e=4, |v|=2"
algorithm,,,,,,,,,
cd_A,1.0,1.00,1.0,0.50,0.80,1.0,0.5,1.0,0.9
cd_B,1.0,0.55,1.0,0.20,0.65,1.0,0.2,0.9,0.7
cd_BOmega,1.0,1.00,1.0,0.35,0.75,1.0,0.5,1.0,0.9
